# Silver — Customers (SCD1)
**GlobalMart Orchestration Lab**

| | |
|---|---|
| **Source** | `{catalog}.bronze.customers` |
| **Target** | `{catalog}.silver.customers` |
| **SCD Type** | SCD1 — latest profile always overwrites the previous one |
| **Depends on** | Bronze Customers task must complete first |

Same investigate-then-decide shape as Day 5's Silver notebooks — read, dedup, DQ scan, transform, MERGE.

## Step 1 — Setup

In [ ]:
from pyspark.sql.functions import col, row_number, desc, trim, lower, concat_ws, to_date, current_timestamp
from pyspark.sql.window import Window
from delta.tables import DeltaTable

dbutils.widgets.text('catalog',       'your_catalog')
dbutils.widgets.text('source_schema', 'bronze')
dbutils.widgets.text('target_schema', 'silver')

CATALOG       = dbutils.widgets.get('catalog')
SOURCE_SCHEMA = dbutils.widgets.get('source_schema')
TARGET_SCHEMA = dbutils.widgets.get('target_schema')

SOURCE_TABLE = f'{CATALOG}.{SOURCE_SCHEMA}.customers'
TABLE        = f'{CATALOG}.{TARGET_SCHEMA}.customers'

print(f'Source: {SOURCE_TABLE}')
print(f'Target: {TABLE}')

## Step 2 — Read &amp; Deduplicate from Bronze
Bronze accumulates every batch via Autoloader append — by the second run it holds rows from both loads. Deduplicate by `customer_id`, keeping the latest row by `_ingested_at`, so the MERGE below always sees the freshest profile per customer.

In [ ]:
bronze_df = spark.table(SOURCE_TABLE)
print(f"Bronze rows (all batches): {bronze_df.count():,}")

dedup_window = Window.partitionBy("customer_id").orderBy(desc("_ingested_at"))
deduped_df = (
    bronze_df
    .withColumn("_rn", row_number().over(dedup_window))
    .filter(col("_rn") == 1)
    .drop("_rn")
)
print(f"After dedup (latest per customer_id): {deduped_df.count():,}")

## Step 3 — DQ Scan: Investigate Before Deciding
Same rule as Day 5 and the supply-chain assessment: don't quarantine on sight, look first. Tag every row with its issue, then decide Fix vs. Flag vs. Quarantine per issue — not per table.

In [ ]:
from pyspark.sql.functions import when, lit

dq_df = deduped_df.withColumn(
    "_dq_issue",
    when(col("customer_id").isNull(),                                  lit("NULL_CUSTOMER_ID"))
    .when(col("email").isNull() | (trim(col("email")) == ""),          lit("MISSING_EMAIL"))
    .when(~col("email").contains("@"),                                 lit("INVALID_EMAIL_FORMAT"))
    .otherwise(lit(None))
)

dq_df.groupBy("_dq_issue").count().orderBy(desc("count")).display()

## Step 4 — Decision Per Issue

| Issue | Decision | Why |
|---|---|---|
| `NULL_CUSTOMER_ID` | **Quarantine** | No usable key — can't MERGE, can't join downstream in Gold |
| `MISSING_EMAIL` / `INVALID_EMAIL_FORMAT` | **Flag, keep the row** | The customer record is still usable (name, city, state) — email is one field, not the whole row. Downstream can decide whether to email this customer, not whether they exist. |

In [ ]:
quarantine_df = dq_df.filter(col("_dq_issue") == "NULL_CUSTOMER_ID")
clean_df      = dq_df.filter((col("_dq_issue").isNull()) | (col("_dq_issue") != "NULL_CUSTOMER_ID"))

print(f"Quarantined (no usable key) : {quarantine_df.count():,}")
print(f"Proceeding to Silver        : {clean_df.count():,}  (flagged-but-kept rows included)")

## Step 5 — Transform

In [ ]:
silver_df = (
    clean_df
    .withColumn("full_name",  concat_ws(" ", col("first_name"), col("last_name")))
    .withColumn("created_at", to_date(col("created_at")))
    .withColumn("_silver_updated_at", current_timestamp())
    .select(
        "customer_id", "full_name", "first_name", "last_name",
        "email", "city", "state", "created_at",
        "_dq_issue", "_silver_updated_at"
    )
)
print(f"Rows ready to MERGE: {silver_df.count():,}")

## Step 6 — Create Silver Table (first run only)

In [ ]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{TARGET_SCHEMA}")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TABLE} (
        customer_id STRING,
        full_name   STRING,
        first_name  STRING,
        last_name   STRING,
        email       STRING,
        city        STRING,
        state       STRING,
        created_at  DATE,
        _dq_issue   STRING,
        _silver_updated_at TIMESTAMP
    )
    USING DELTA
""")
print(f"Table ready: {TABLE}")

## Step 7 — SCD1 MERGE
Update if `customer_id` already exists (profile may have changed — this is exactly how batch 2's `CUST-002` email change gets picked up), insert if new. Same code runs unchanged on batch 1 and batch 2.

In [ ]:
target = DeltaTable.forName(spark, TABLE)

(target.alias("t")
    .merge(silver_df.alias("s"), "t.customer_id = s.customer_id")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)
print("MERGE complete")

## Step 8 — Verify

In [ ]:
result = spark.table(TABLE)
print(f"Total rows in {TABLE}: {result.count():,}")
result.display()